# Resistance filtering

Enrichment of intrinsically resistant (high-MIC) taxa among the non-focal species of the
16-species community, and the species-level relation between MIC class and antibiotic response.

Outputs: Figures S8-S9, Tables S10-S11.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%run 00_setup_and_functions.ipynb

## Table S10: high-MIC share and MIC-class score of the non-focal community
Within each history, every non-zero dose is compared with the no-antibiotic value of the same
landscape (exact sign-flip test, Holm across the three doses); between histories, landscapes are
compared at each dose (exact allocation test, Holm across the four doses).

In [2]:
mic_landscape, enrichment, history_contrasts = resistance_filtering()
metric_labels = {"high_MIC_share_excl_focal": "High-MIC non-focal abundance share",
                 "community_weighted_MIC_score_excl_focal": "Non-focal community-weighted MIC-class score"}
table_s10 = enrichment.assign(history=enrichment.community.map({"ANC_SIXTEEN": "ancestral", "EVO_SIXTEEN": "resistance-evolved"}),
                             metric=enrichment.metric.map(metric_labels))
table_s10 = table_s10[["history", "metric", "streptomycin", "no_antibiotic_mean", "dose_mean", "dose_minus_no_antibiotic",
                     "p_exact_paired", "p_holm_across_nonzero_doses"]]
between = (history_contrasts.assign(metric=history_contrasts.metric.map(metric_labels))
           [["metric", "streptomycin", "evolved_minus_ancestral", "p_holm_across_doses"]]
           .rename(columns={"evolved_minus_ancestral": "between_history_difference", "p_holm_across_doses": "p_holm_between_histories"}))
table_s10 = table_s10.merge(between, on=["metric", "streptomycin"], how="left")
table_s10.to_csv(TABLE_DIR / "Table_S10_resistance_filtering.csv", index=False)
history_contrasts

,metric,streptomycin,ancestral_mean,evolved_mean,evolved_minus_ancestral,p_exact,p_holm_across_doses
0,high_MIC_share_excl_focal,0.0,0.257459,0.194222,-0.063237,0.000155,0.000622
1,high_MIC_share_excl_focal,2.0,0.165885,0.125111,-0.040774,0.000466,0.001399
2,high_MIC_share_excl_focal,20.0,0.686832,0.777358,0.090526,0.002020,0.002020
3,high_MIC_share_excl_focal,200.0,0.852195,0.799593,-0.052603,0.000622,0.001399
4,community_weighted_MIC_score_excl_focal,0.0,1.813665,1.615529,-0.198135,0.000155,0.000622
5,community_weighted_MIC_score_excl_focal,2.0,1.614501,1.504774,-0.109727,0.001399,0.002486
6,community_weighted_MIC_score_excl_focal,20.0,3.228340,3.475610,0.247270,0.001243,0.002486
7,community_weighted_MIC_score_excl_focal,200.0,3.694543,3.529035,-0.165508,0.000155,0.000622


## Table S11: MIC class and species-level antibiotic response
Spearman correlation between ancestral MIC class and the mean CLR response of each species,
with 200,000 permutations of the MIC labels; Holm across the three doses.

In [3]:
dclr_rep, _ = whole_community_clr_response()
species_response = species_clr_responses(dclr_rep)
rows = []
for community in ["ANC_SIXTEEN", "EVO_SIXTEEN"]:
    family = []
    for dose in NONZERO_DOSES:
        sub = species_response.loc[(species_response.community == community) & (species_response.streptomycin == dose)]
        rho, p = mic_permutation_p(sub.MIC_score, sub.mean_delta_clr,
                                   seed=RANDOM_SEED + int(dose) + (1000 if community.startswith("EVO") else 0))
        family.append({"history": "ancestral" if community.startswith("ANC") else "resistance-evolved", "streptomycin": dose,
                       "spearman_rho_MIC_vs_response": rho, "p_permutation": p, "n_species": len(sub), "n_permutations": 200000})
    for r, q in zip(family, holm([r["p_permutation"] for r in family])):
        r["p_holm_across_doses"] = q
    rows += family
table_s11 = pd.DataFrame(rows)[["history", "streptomycin", "spearman_rho_MIC_vs_response", "p_permutation",
                               "p_holm_across_doses", "n_species", "n_permutations"]]
table_s11.to_csv(TABLE_DIR / "Table_S11_MIC_response_sensitivity.csv", index=False)
table_s11

,history,streptomycin,spearman_rho_MIC_vs_response,p_permutation,p_holm_across_doses,n_species,n_permutations
0,ancestral,2.0,-0.097996,0.801491,0.801491,9,200000
1,ancestral,20.0,0.775058,0.022370,0.067110,9,200000
2,ancestral,200.0,0.677062,0.055615,0.111229,9,200000
3,resistance-evolved,2.0,0.138321,0.726161,0.726161,9,200000
4,resistance-evolved,20.0,0.774597,0.023740,0.071220,9,200000
5,resistance-evolved,200.0,0.507176,0.165944,0.331888,9,200000


## Figure S8: high-MIC share and MIC-class score of the non-focal community (Table S10)

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(150 * MM, 78 * MM), sharex=True)
for ax, metric, ylabel, scale in [(axes[0], "high_MIC_share_excl_focal", "Community abundance in high-MIC taxa (%)", 100.0),
                                  (axes[1], "community_weighted_MIC_score_excl_focal", "Community-weighted MIC-class score", 1.0)]:
    for community, color in [("ANC_SIXTEEN", ANCESTRAL_COLOR), ("EVO_SIXTEEN", EVOLVED_COLOR)]:
        pvt = (mic_landscape.loc[mic_landscape.community == community]
               .pivot(index="replicate", columns="streptomycin", values=metric).reindex(columns=STREPTOMYCIN_VALUES))
        for xi, dose in enumerate(STREPTOMYCIN_VALUES):
            replicate_mean_ci(ax, xi + (-0.06 if community == "ANC_SIXTEEN" else 0.06), scale * pvt[dose].to_numpy(float), color,
                              seed=6000 + xi + (0 if community == "ANC_SIXTEEN" else 50), jitter_width=0.02)
    ax.set_xticks(range(4)); ax.set_xticklabels([f"{x:g}" for x in STREPTOMYCIN_VALUES])
    ax.set_xlabel(STREP_AXIS_LABEL); ax.set_ylabel(ylabel, fontsize=6.0)
axes[0].set_ylim(0, 100); axes[1].set_ylim(0, 4.1)
fig.legend(handles=history_legend_handles(), loc="lower center", bbox_to_anchor=(0.5, 0.00), ncol=2, frameon=False, fontsize=5.4)
fig.subplots_adjust(left=0.11, right=0.99, bottom=0.28, top=0.95, wspace=0.36)
save_figure(fig, "Figure_S8_high_MIC_enrichment")

## Figure S9: MIC class and species-level antibiotic response (Table S11)

In [5]:
fig, axes = plt.subplots(2, 3, figsize=(183 * MM, 115 * MM), sharey=True)
for row, c in enumerate(["ANC_SIXTEEN", "EVO_SIXTEEN"]):
    for col, dose in enumerate(NONZERO_DOSES):
        ax = axes[row, col]
        sub = species_response.loc[(species_response.community == c) & (species_response.streptomycin == dose)].sort_values("MIC_score")
        ax.scatter(sub.MIC_score, sub.mean_delta_clr, s=16, facecolor="white", edgecolor=ANCESTRAL_COLOR if row == 0 else EVOLVED_COLOR, lw=0.8)
        ax.axhline(0, color="0.72", lw=0.6)
        ax.set_xticks(range(5)); ax.set_xticklabels(MIC_LABEL_ORDER)
        if row == 1:
            ax.set_xlabel("Ancestral MIC class (µg ml$^{-1}$)")
        if col == 0:
            ax.set_ylabel("Mean antibiotic response (ΔCLR)")
            ax.text(-0.34, 0.5, ("Community containing ancestral" if row == 0 else "Community containing resistance-evolved")
                    + "\n$\\it{Aeromonas\\ caviae}$", transform=ax.transAxes, rotation=90, va="center", ha="center", fontsize=5.5)
        if row == 0:
            ax.text(0.5, 1.025, f"{dose:g} µg ml$^{{-1}}$", transform=ax.transAxes, ha="center", va="bottom", fontsize=6.4)
fig.subplots_adjust(left=0.14, right=0.99, bottom=0.13, top=0.94, hspace=0.22, wspace=0.18)
save_figure(fig, "Figure_S9_MIC_response")